# Immigration Discourse Dataset — Corpus Analysis

Statistics and visualizations for an ACL submission corpus description / response to reviewer feedback.

**Corpus:** ~126K immigration news articles, GDELT-sourced, scraped via a custom pipeline. Stored as 100 JSONL shards on `s3://immigration-discourse-dataset/data/`.

Sections:
1. Load (with parquet cache)
2. Dataset summary (volume, time, sources, length, missingness)
3. Immigration term analysis (counts, co-occurrence, over time, by source, heatmap)
4. Source diversity (long tail, thresholds, partisan flags)
5. Text-quality checks (length, short articles, duplicates, eyeball samples)
6. Data-statement-ready summary

## 0. Setup

In [ ]:
import os
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from immigration_corpus import load_data

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

CACHE_PARQUET = Path('dataset_cache.parquet')
print('Setup OK. Parquet cache target:', CACHE_PARQUET.resolve())

## 1. Load Full Dataset

First run: streams all 100 JSONL shards from S3 (~2.8 GB) and writes a local parquet snapshot.
Subsequent runs: reads parquet directly (fast).

In [ ]:
t0 = time.time()
if CACHE_PARQUET.exists():
    print(f'Loading from local parquet cache: {CACHE_PARQUET}')
    df = pd.read_parquet(CACHE_PARQUET)
else:
    print('No local cache - downloading all 100 shards from S3...')
    df = load_data()  # loads files 0..99 with progress prints
    print(f'\nWriting parquet cache to {CACHE_PARQUET} ...')
    # authors is a list of strings - keep as object; parquet handles it.
    df.to_parquet(CACHE_PARQUET, index=False)
elapsed = time.time() - t0
print(f'\nLoaded {len(df):,} articles in {elapsed:,.1f}s')
print('Columns:', list(df.columns))
df.head(3)

In [ ]:
# Normalize publish_date once for the rest of the notebook.
df['publish_date'] = pd.to_datetime(df['publish_date'], errors='coerce', utc=True)
df['year']  = df['publish_date'].dt.year
df['month'] = df['publish_date'].dt.to_period('M')

# Pre-compute text length once.
df['text_len_chars'] = df['text'].fillna('').str.len()
df['text_len_words'] = df['text'].fillna('').str.split().str.len()

print('Date parse success rate:', f"{df['publish_date'].notna().mean()*100:.2f}%")
print('Year range observed:', int(df['year'].min(skipna=True)), 'to', int(df['year'].max(skipna=True)))

## 2. Dataset Summary

In [ ]:
summary = {
    'total_articles':  len(df),
    'unique_sources':  df['source'].nunique(),
    'earliest':        df['publish_date'].min(),
    'latest':          df['publish_date'].max(),
    'mean_chars':      df['text_len_chars'].mean(),
    'median_chars':    df['text_len_chars'].median(),
    'mean_words':      df['text_len_words'].mean(),
    'median_words':    df['text_len_words'].median(),
}
for k, v in summary.items():
    if isinstance(v, float):
        print(f'{k:>18}: {v:,.1f}')
    else:
        print(f'{k:>18}: {v}')

In [ ]:
# Year-by-year article distribution
by_year = df['year'].value_counts().sort_index()
by_year_df = by_year.rename_axis('year').reset_index(name='articles')
print(by_year_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(by_year.index.astype(int).astype(str), by_year.values, color='steelblue')
ax.set_title('Articles per Year')
ax.set_xlabel('Year'); ax.set_ylabel('Articles')
for i, v in enumerate(by_year.values):
    ax.text(i, v, f'{int(v):,}', ha='center', va='bottom', fontsize=8)
plt.xticks(rotation=45)
plt.tight_layout(); plt.show()

In [ ]:
# Month-by-month, most recent 2 years observed in data
max_year = int(df['year'].dropna().max())
recent = df[df['year'].isin([max_year - 1, max_year])].copy()
by_month = recent['month'].value_counts().sort_index()
by_month_df = by_month.rename_axis('month').reset_index(name='articles')
print(by_month_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(13, 4))
ax.bar([str(m) for m in by_month.index], by_month.values, color='darkorange')
ax.set_title(f'Articles per Month - {max_year-1} & {max_year}')
ax.set_xlabel('Month'); ax.set_ylabel('Articles')
plt.xticks(rotation=60)
plt.tight_layout(); plt.show()

In [ ]:
# Top 30 sources
top30 = df['source'].value_counts().head(30)
top30_df = top30.rename_axis('source').reset_index(name='articles')
top30_df['pct_of_corpus'] = (top30_df['articles'] / len(df) * 100).round(2)
print(top30_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(top30.index[::-1], top30.values[::-1], color='teal')
ax.set_title('Top 30 Sources by Article Count')
ax.set_xlabel('Articles')
plt.tight_layout(); plt.show()

In [ ]:
# Missing data rates
def missing_rate(s):
    n = len(s)
    if s.dtype == object:
        empty = s.apply(lambda v: v is None or (isinstance(v, float) and np.isnan(v))
                        or (isinstance(v, str) and v.strip() == '')
                        or (isinstance(v, list) and len(v) == 0))
    else:
        empty = s.isna()
    miss = int(empty.sum())
    return miss, miss / n * 100

rows = []
for col in ['source', 'url', 'title', 'header', 'text', 'authors', 'publish_date']:
    miss, pct = missing_rate(df[col])
    rows.append({'field': col, 'missing': miss, 'pct_missing': round(pct, 3)})
missing_df = pd.DataFrame(rows)
print(missing_df.to_string(index=False))

## 3. Immigration Term Analysis

In [ ]:
# Compile case-insensitive word-boundary regexes for each term.
# Word boundaries avoid matching e.g. 'undocumented immigrants' inside hyphenated words.
TERMS = [
    'illegal alien', 'illegal aliens',
    'undocumented immigrant', 'undocumented immigrants',
    'illegal immigrant', 'illegal immigrants',
]
patterns = {t: re.compile(rf'\b{re.escape(t)}\b', re.IGNORECASE) for t in TERMS}

# Search across title + header + text so a term doesn't get missed if it only appears in the headline.
haystack = (df['title'].fillna('') + ' \n ' + df['header'].fillna('') + ' \n ' + df['text'].fillna(''))

print('Scanning corpus for term presence (this may take ~30s)...')
term_present = {}
for t, pat in patterns.items():
    term_present[t] = haystack.str.contains(pat, na=False)
    n = int(term_present[t].sum())
    print(f'  {t:>27}: {n:>7,} articles ({n/len(df)*100:.2f}%)')

term_df = pd.DataFrame(term_present)

In [ ]:
# Group singular/plural
groups = {
    'illegal_alien':           term_df['illegal alien'] | term_df['illegal aliens'],
    'undocumented_immigrant':  term_df['undocumented immigrant'] | term_df['undocumented immigrants'],
    'illegal_immigrant':       term_df['illegal immigrant'] | term_df['illegal immigrants'],
}
group_df = pd.DataFrame(groups)

group_counts = group_df.sum().rename('articles').to_frame()
group_counts['pct_of_corpus'] = (group_counts['articles'] / len(df) * 100).round(3)
print(group_counts)

# Co-occurrence: BOTH an illegal-alien(s) term AND an undocumented-immigrant(s) term in the same article.
co_aa_ui = int((groups['illegal_alien'] & groups['undocumented_immigrant']).sum())
print(f"\nArticles containing BOTH 'illegal alien(s)' AND 'undocumented immigrant(s)': {co_aa_ui:,} "
      f"({co_aa_ui/len(df)*100:.3f}% of corpus)")

# Full pairwise co-occurrence matrix for the three groups.
co = pd.DataFrame(index=group_df.columns, columns=group_df.columns, dtype=int)
for a in group_df.columns:
    for b in group_df.columns:
        co.loc[a, b] = int((group_df[a] & group_df[b]).sum())
print('\nCo-occurrence (rows AND cols):')
print(co)

In [ ]:
# Term usage over time (year-by-year frequency for the three grouped terms)
time_df = group_df.copy()
time_df['year'] = df['year'].values
yearly = time_df.dropna(subset=['year']).groupby('year').sum().astype(int)
yearly_pct = yearly.div(df.dropna(subset=['year']).groupby('year').size(), axis=0) * 100

print('Absolute counts by year:')
print(yearly)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
yearly.plot(kind='line', marker='o', ax=axes[0])
axes[0].set_title('Term Usage Over Time (absolute)')
axes[0].set_ylabel('Articles containing term')
axes[0].set_xlabel('Year'); axes[0].legend(title='Term group')
yearly_pct.plot(kind='line', marker='o', ax=axes[1])
axes[1].set_title('Term Usage Over Time (% of that year)')
axes[1].set_ylabel('% of articles in year')
axes[1].set_xlabel('Year'); axes[1].legend(title='Term group')
plt.tight_layout(); plt.show()

In [ ]:
# Term usage by source (top 20 sources by total article count)
top20_sources = df['source'].value_counts().head(20).index.tolist()

by_source = group_df.copy()
by_source['source'] = df['source'].values
by_source = by_source[by_source['source'].isin(top20_sources)]

src_totals = by_source.groupby('source').size().rename('total_articles')
src_counts = by_source.groupby('source')[list(group_df.columns)].sum()
src_pct = (src_counts.div(src_totals, axis=0) * 100).round(2)
src_pct = src_pct.loc[top20_sources]  # preserve order by corpus count
src_pct.insert(0, 'total_articles', src_totals.loc[top20_sources].astype(int))
print('Per-source term usage (% of that source\'s articles):')
print(src_pct)

In [ ]:
# Heatmap: % of each top-20 source's articles using each term group
heat = src_pct.drop(columns=['total_articles'])
fig, ax = plt.subplots(figsize=(8, 9))
sns.heatmap(heat, annot=True, fmt='.1f', cmap='YlOrRd', cbar_kws={'label': '% of source articles'}, ax=ax)
ax.set_title('Immigration term usage by top-20 source (% of that source\'s articles)')
plt.tight_layout(); plt.show()

## 4. Source Diversity

In [ ]:
src_counts_all = df['source'].value_counts()

print(f'Total unique sources: {len(src_counts_all):,}')
for thresh in [10, 50, 100, 500, 1000, 5000]:
    n = int((src_counts_all >= thresh).sum())
    print(f'  sources with >= {thresh:>5,} articles: {n:>5,}')

# Long-tail histogram (log scale)
fig, ax = plt.subplots(figsize=(11, 4))
ax.hist(src_counts_all.values, bins=80, color='slateblue', edgecolor='white')
ax.set_yscale('log')
ax.set_xlabel('Articles per source')
ax.set_ylabel('Number of sources (log)')
ax.set_title('Distribution of articles per source (long tail)')
plt.tight_layout(); plt.show()

In [ ]:
# Rough partisan flagging for well-known outlets. Not a classifier - intentionally narrow.
LEAN = {
    'right': ['foxnews.com', 'breitbart.com', 'dailycaller.com', 'nationalreview.com',
              'theblaze.com', 'nypost.com', 'washingtontimes.com', 'townhall.com',
              'theamericanconservative.com', 'dailywire.com', 'newsmax.com', 'theepochtimes.com',
              'theFederalist.com', 'thefederalist.com', 'pjmedia.com', 'redstate.com'],
    'left':  ['huffpost.com', 'msnbc.com', 'cnn.com', 'motherjones.com', 'thenation.com',
              'salon.com', 'vox.com', 'slate.com', 'theguardian.com', 'theintercept.com',
              'jacobin.com', 'commondreams.org', 'truthout.org', 'rawstory.com'],
    'center':['reuters.com', 'apnews.com', 'bbc.com', 'bbc.co.uk', 'npr.org', 'pbs.org',
             'cbsnews.com', 'abcnews.go.com', 'nbcnews.com', 'usatoday.com', 'axios.com',
             'thehill.com', 'politico.com', 'bloomberg.com', 'wsj.com', 'nytimes.com',
             'washingtonpost.com'],
}
lookup = {s: lean for lean, srcs in LEAN.items() for s in srcs}
df['lean'] = df['source'].map(lookup).fillna('unclassified')

lean_counts = df['lean'].value_counts()
lean_pct = (lean_counts / len(df) * 100).round(2)
lean_summary = pd.DataFrame({'articles': lean_counts, 'pct_of_corpus': lean_pct})
print(lean_summary)

# Term usage by lean (excluding unclassified for readability)
lean_term = group_df.copy()
lean_term['lean'] = df['lean'].values
lean_pct_table = (lean_term.groupby('lean').mean() * 100).round(2)
print('\nTerm usage by lean (% of that lean\'s articles):')
print(lean_pct_table)

## 5. Text Quality Checks

In [ ]:
# Text-length distribution (clipped at 99th percentile for plot readability)
p99 = df['text_len_chars'].quantile(0.99)
fig, ax = plt.subplots(figsize=(11, 4))
ax.hist(df['text_len_chars'].clip(upper=p99), bins=80, color='seagreen', edgecolor='white')
ax.set_xlabel(f'Article length (chars, clipped at p99 = {int(p99):,})')
ax.set_ylabel('Articles')
ax.set_title('Distribution of article text length')
plt.tight_layout(); plt.show()

print('Char-length percentiles:')
print(df['text_len_chars'].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(1))

In [ ]:
# Very short articles
short_mask = df['text_len_chars'] < 100
n_short = int(short_mask.sum())
print(f'Articles with <100 chars of text: {n_short:,} ({n_short/len(df)*100:.2f}%)')

for thresh in [0, 50, 100, 250, 500]:
    n = int((df['text_len_chars'] < thresh).sum()) if thresh > 0 else int((df['text_len_chars'] == 0).sum())
    label = f'== 0' if thresh == 0 else f'< {thresh}'
    print(f'  text length {label:>7}: {n:>6,} ({n/len(df)*100:.2f}%)')

In [ ]:
# Potential duplicates: same (source, title)
dup_key = df[['source', 'title']].fillna('')
dup_mask = dup_key.duplicated(keep=False) & (dup_key['title'].str.strip() != '')
n_dup_rows = int(dup_mask.sum())
n_dup_groups = int(dup_key[dup_mask].drop_duplicates().shape[0])
print(f'Articles in a (source, title) duplicate group: {n_dup_rows:,} ({n_dup_rows/len(df)*100:.2f}%)')
print(f'Number of distinct duplicate (source, title) groups: {n_dup_groups:,}')

# Top duplicate offenders
top_dups = (dup_key[dup_mask].value_counts().head(10)
            .rename_axis(['source', 'title']).reset_index(name='occurrences'))
print('\nTop 10 (source, title) duplicates:')
print(top_dups.to_string(index=False))

In [ ]:
# 5 random article samples (truncated text) for eyeball QA
samples = df.sample(n=5, random_state=42)
for i, row in samples.iterrows():
    print('=' * 80)
    print(f"source: {row['source']}")
    print(f"date  : {row['publish_date']}")
    print(f"title : {row['title']}")
    txt = (row['text'] or '')[:500].replace('\n', ' ')
    print(f"text  : {txt}{'...' if row['text_len_chars'] > 500 else ''}")
    print()

## 6. Data-Statement Summary (copy/paste for paper)

In [ ]:
from IPython.display import Markdown, display

earliest = summary['earliest']
latest   = summary['latest']
earliest_s = earliest.strftime('%Y-%m-%d') if pd.notna(earliest) else 'n/a'
latest_s   = latest.strftime('%Y-%m-%d')   if pd.notna(latest)   else 'n/a'

g = group_counts['articles'].to_dict()
g_pct = group_counts['pct_of_corpus'].to_dict()

top5_sources = src_counts_all.head(5)
top5_str = ', '.join(f"{s} ({c:,})" for s, c in top5_sources.items())

md = f"""### Corpus data statement (auto-generated)

**Size.** The corpus contains **{summary['total_articles']:,} articles** drawn from **{summary['unique_sources']:,} unique news source domains**.

**Temporal coverage.** Articles are dated between **{earliest_s}** and **{latest_s}**. Publish-date parsing succeeded for **{df['publish_date'].notna().mean()*100:.2f}%** of articles.

**Collection methodology.** Sources were identified via the GDELT global news index using immigration-related queries scoped to mentions of the United States. Article HTML was retrieved and parsed with a custom scraping pipeline that extracted source, URL, title, header, full body text, authors, and publish date.

**Language.** English-language news only (inherited from the GDELT query and the targeted U.S.-mentioning domains).

**Top sources.** The five largest contributors are: {top5_str}.

**Article length.** Mean **{summary['mean_chars']:,.0f}** chars / **{summary['mean_words']:,.0f}** words; median **{summary['median_chars']:,.0f}** chars / **{summary['median_words']:,.0f}** words. Articles with <100 chars of body text: **{n_short:,} ({n_short/len(df)*100:.2f}%)**.

**Key term distributions (article-level, case-insensitive, word-bounded; counts an article once if any singular/plural form appears in title, header, or body).**

- `illegal alien(s)`: **{int(g['illegal_alien']):,}** articles ({g_pct['illegal_alien']:.2f}%)
- `undocumented immigrant(s)`: **{int(g['undocumented_immigrant']):,}** articles ({g_pct['undocumented_immigrant']:.2f}%)
- `illegal immigrant(s)`: **{int(g['illegal_immigrant']):,}** articles ({g_pct['illegal_immigrant']:.2f}%)
- Articles using BOTH `illegal alien(s)` AND `undocumented immigrant(s)`: **{co_aa_ui:,}** ({co_aa_ui/len(df)*100:.3f}%)

**Duplicates.** **{n_dup_rows:,}** articles ({n_dup_rows/len(df)*100:.2f}%) share a `(source, title)` with at least one other article, spread across **{n_dup_groups:,}** distinct duplicate groups.

**Source-lean flagging (illustrative only, not a classification).** Out of {len(df):,} articles, {int(lean_counts.get('right', 0)):,} are from outlets we flagged as right-leaning, {int(lean_counts.get('left', 0)):,} as left-leaning, {int(lean_counts.get('center', 0)):,} as centrist mainstream, and {int(lean_counts.get('unclassified', 0)):,} unclassified. This labeling is restricted to a hand-picked set of well-known U.S. outlets and is meant to motivate diversity, not to support per-source partisanship claims.
"""
display(Markdown(md))

In [ ]:
# Also write the data statement to disk for easy paper copy/paste.
Path('data_statement.md').write_text(md, encoding='utf-8')
print('Wrote data_statement.md')